In [2]:
import pandas as pd
import pickle
import polars as pl
from pathlib import Path

In [3]:
def load_parquet_dir(path: str | Path) -> pl.DataFrame:
    """
    Load all parquet files in a directory into a single Polars DataFrame.

    Parameters
    ----------
    path : str | Path
        Path to the directory containing parquet files.

    Returns
    -------
    pl.DataFrame
        Concatenated dataframe containing all parquet data.
    """

    path = Path(path)

    parquet_files = sorted(path.glob("*.parquet"))

    if not parquet_files:
        raise FileNotFoundError(f"No parquet files found in: {path}")

    dfs = [pl.read_parquet(file) for file in parquet_files]

    return pl.concat(dfs, how="vertical")

In [4]:
# read open targets associations
base = "/home/ubuntu/Data/"
ot_associations = load_parquet_dir(base+"open_targets_26_09/association_by_datasource_direct/")

In [5]:
ot_associations.head(2)

diseaseId,targetId,aggregationType,aggregationValue,associationScore,evidenceCount,timeseries,currentNovelty
str,str,str,str,f64,i64,list[struct[4]],f64
"""EFO_0000180""","""ENSG00000020633""","""datasourceId""","""europepmc""",0.030415,1,"[{2025,0.030415,0.0,null}, {2012,0.0,0.0,null}, … {2009,0.0,0.0,null}]",0.0
"""EFO_0000180""","""ENSG00000081985""","""datasourceId""","""europepmc""",0.012166,1,"[{2022,0.012166,0.012136,1}, {2019,0.0,0.0,null}, … {2017,0.0,0.0,null}]",0.00145


In [6]:
top_genes = pd.read_csv("../output/top_1perc_genes.csv", index_col=0)
top_genes.head(2)

,ppr_importance_string,zscore_string,rank_string
PTPRD,0.031839,49.162583,1
NTM,0.029002,44.771989,2


In [7]:
# get disease information 
ot_disease = load_parquet_dir(base+"open_targets_26_09/disease/")
ot_disease.head(1)

id,code,name,description,dbXRefs,parents,exactSynonyms,relatedSynonyms,narrowSynonyms,broadSynonyms,obsoleteTerms,obsoleteXRefs,children,ancestors,therapeuticAreas,descendants,ontology,synonyms
str,str,str,str,list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],list[str],struct[3],struct[4]
"""GO_0000050""","""http://purl.obolibrary.org/obo…","""urea cycle""","""The sequence of reactions by w…","[""Reactome:R-HSA-70635"", ""Wikipedia:Urea_cycle""]","[""GO_0009058""]","[""ornithine cycle"", ""urea biosynthesis"", ""urea biosynthetic process""]",[],[],[],[],[],[],"[""GO_0009058"", ""GO_0008152"", … ""GO_0008150""]","[""GO_0008150""]",[],"{false,false,{""http://purl.obolibrary.org/obo/GO_0000050"",""GO_0000050""}}","{[""ornithine cycle"", ""urea biosynthesis"", ""urea biosynthetic process""],[],[],[]}"


In [8]:
# get genes information for OT
ot_genes = load_parquet_dir(base+"open_targets_26_09/target/")
ot_genes.head(1)


id,approvedSymbol,biotype,genomicLocation,approvedName,go,hallmarks,synonyms,symbolSynonyms,nameSynonyms,functionDescriptions,subcellularLocations,targetClass,obsoleteSymbols,obsoleteNames,constraint,proteinIds,dbXrefs,pathways,tss
str,str,str,struct[4],str,list[struct[6]],struct[2],list[struct[2]],list[struct[2]],list[struct[2]],list[str],list[struct[5]],list[struct[3]],list[struct[2]],list[struct[2]],list[struct[10]],list[struct[2]],list[struct[2]],list[struct[3]],i64
"""ENSG00000083454""","""P2RX5""","""protein_coding""","{""17"",3669763,3696398,-1}","""purinergic receptor P2X 5""","[{""GO:0033198"",""GO_REF:0000107"",""IEA"",""P"",""Q93086"",""ECO:0000256""}, {""GO:0050850"",""PMID:17895406"",""NAS"",""P"",""Q93086"",""ECO:0000303""}, … {""GO:0004931"",""GO_REF:0000002"",""IEA"",""F"",""Q93086"",""ECO:0000256""}]",null,"[{""P2X purinoceptor 5"",""uniprot""}, {""P2X5"",""uniprot""}, … {""P2RX5"",""NCBI_entrez""}]","[{""P2X5"",""uniprot""}, {""P2RX5"",""uniprot""}, … {""P2RX5"",""NCBI_entrez""}]","[{""P2X purinoceptor 5"",""uniprot""}, {""ATP receptor"",""uniprot""}, … {""purinergic receptor P2X, ligand gated ion channel, 5"",""NCBI_entrez""}]","[""[Isoform 6]: ATP-gated nonselective transmembrane cation channel permeable to potassium, sodium and calcium (PubMed:12761352). Unlike other P2RX receptors, the P2X5 receptor is also permeable to chloride (PubMed:12761352). May play a supporting role in the inflammatory response (By similarity). {ECO:0000250|UniProtKB:Q91VE2, ECO:0000269|PubMed:12761352}."", ""[Isoform 1]: Non-functional. .""]","[{""Cell membrane ; Multi-pass membrane protein"",""uniprot"",""SL-0039"",""Cellular component"",""Isoform 6""}, {""Cytosol"",""HPA_main"",""SL-0091"",""Cytosol"",null}, {""Plasma membrane"",""HPA_main"",""SL-0039"",""Cell membrane"",null}]","[{1047,""Ligand-gated ion channel"",""l2""}, {1047,""P2X receptor"",""l3""}, {1047,""Ion channel"",""l1""}]",[],"[{""purinergic receptor P2X, ligand-gated ion channel, 5"",""HGNC""}]","[{""syn"",0.016201,226.410004,226,0.9982,0.89572,1.1145,null,null,null}, {""mis"",-0.2948,549.789978,566,1.0295,0.96111,1.1036,null,null,null}, {""lof"",6.4572e-14,54.077999,46,0.85063,0.67162,1.0876,12124,6,3}]","[{""Q93086"",""uniprot_swissprot""}, {""B4DEG2"",""uniprot_trembl""}, … {""Q9NZV0"",""uniprot_obsolete""}]","[{""8536"",""HGNC""}, {""CHEMBL4942"",""ChEMBL""}, … {""IPR053792"",""InterPro""}]","[{""R-HSA-418346"",""Platelet homeostasis"",""Hemostasis""}, {""R-HSA-139853"",""Elevation of cytosolic Ca2+ levels"",""Hemostasis""}]",3696155


# Step 1

Filter OT for interesting diseases, i.e. all with this following parents (including themselves):

- Pain: HP_0012531
- Neuropathy: EFO_0004149
- Fibromyalgia: MONDO_0005546

In [9]:
target_set = {'HP_0012531', 'EFO_0004149', 'MONDO_0005546'}

# filter for (OR)
# id in target set
# dbXRefs in target set
# parents  in target set
filtered_diseases = ot_disease.filter(
    pl.col("id").is_in(target_set)
    | pl.col("dbXRefs").list.eval(
        pl.element().is_in(target_set)
    ).list.any()
    | pl.col("parents").list.eval(
        pl.element().is_in(target_set)
    ).list.any()
)

In [10]:
filtered_diseases.shape

(54, 18)

In [11]:
# Filter associations
ot_associations_filtered = ot_associations.filter(
    pl.col('diseaseId').is_in(set(filtered_diseases['id']))
)
ot_associations_filtered.head(1)

diseaseId,targetId,aggregationType,aggregationValue,associationScore,evidenceCount,timeseries,currentNovelty
str,str,str,str,f64,i64,list[struct[4]],f64
"""EFO_0004149""","""ENSG00000012061""","""datasourceId""","""europepmc""",0.168996,12,"[{2010,0.133825,0.133494,1}, {2026,0.168996,0.000179,null}, … {2014,0.146118,0.015952,1}]",0.000179


In [12]:
# take gene symbol

In [13]:
pain_assoc = (
    ot_associations_filtered
    .join(
        ot_genes,
        left_on="targetId",
        right_on="id",
        validate="m:1",
        how="left",
    )
    .join(
        ot_disease,
        left_on="diseaseId",
        right_on="id",
        validate="m:1",
        how="left",
    )
    .select(['diseaseId','targetId','approvedSymbol','name','aggregationValue','associationScore','description'])
)

In [16]:
pain_assoc

diseaseId,targetId,approvedSymbol,name,aggregationValue,associationScore,description
str,str,str,str,str,f64,str
"""EFO_0004149""","""ENSG00000012061""","""ERCC1""","""neuropathy""","""europepmc""",0.168996,"""A disorder affecting the nervo…"
"""EFO_0004149""","""ENSG00000035403""","""VCL""","""neuropathy""","""europepmc""",0.022811,"""A disorder affecting the nervo…"
"""EFO_0004149""","""ENSG00000054598""","""FOXC1""","""neuropathy""","""impc""",0.383972,"""A disorder affecting the nervo…"
"""EFO_0004149""","""ENSG00000072310""","""SREBF1""","""neuropathy""","""europepmc""",0.039539,"""A disorder affecting the nervo…"
"""EFO_0004149""","""ENSG00000082701""","""GSK3B""","""neuropathy""","""europepmc""",0.135522,"""A disorder affecting the nervo…"
…,…,…,…,…,…,…
"""EFO_0004149""","""ENSG00000213903""","""LTB4R""","""neuropathy""","""europepmc""",0.018249,"""A disorder affecting the nervo…"
"""EFO_0005762""","""ENSG00000174788""","""PCP2""","""neuropathic pain""","""gwas_credible_sets""",0.351553,"""Chronic pain caused by damage …"
"""MONDO_0004183""","""ENSG00000171298""","""GAA""","""axonal neuropathy""","""europepmc""",0.018249,"""Any nerve disorder affecting t…"


# Step 2

Annotate pain associations for PPR percentile

In [20]:
ppr_result = pd.read_csv("../output/full_ppr.csv", index_col=0)

ppr_result = (
    pl.read_csv("../output/full_ppr.csv")
    .with_columns(
        (
            1 - (pl.col("rank_string") - 1) / (pl.len() - 1)
        ).alias("percentile")
    )
)

ppr_result.head(5)

,ppr_importance_string,zscore_string,rank_string,Gene,percentile
str,f64,f64,i64,str,f64
"""PTPRD""",0.031839,49.162583,1,"""PTPRD""",1.0
"""NTM""",0.029002,44.771989,2,"""NTM""",0.99993
"""SDC1""",0.025801,39.818093,3,"""SDC1""",0.99986
"""LAPTM4A""",0.024126,37.226708,4,"""LAPTM4A""",0.99979
"""KCNQ3""",0.024088,37.166788,5,"""KCNQ3""",0.99972


In [22]:
pain_assoc_ppr = pain_assoc.join(
    ppr_result,
    left_on="approvedSymbol",
    right_on="Gene"
)

In [23]:
pain_assoc_ppr.write_csv("../output/OT_pain_assoc_ppr.csv")